# 2026-11-21 EDA on FOODS, day 2
Messy exploring is allowed here. Anything worth keeping moves into `src/m5food`.

Question for this notebook: *(write today's one question here)*

## Open from yesterday (2026-11-20_EDA_on_FOODS)
- **Alive:** how do we define when an item-store series is alive? At the end of the data, per day, or over its history? Which signals: first sale, last sale, gap longer than N days, price exists?
- **First sale vs launch:** a slow seller's first sale can come weeks after it reached the shelf. How do we tell them apart?
- **Spread threshold:** which gap between stores' first sales counts as "far apart" (7 / 28 / 90 / 180 days), and why?
- **Zero days:** a 1-day zero and a 120-day gap both count as zero days. Same treatment or not?

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import plotly.express as px, plotly.graph_objects as go
from m5food import data, plots
plots.set_style()

## Load
Plain names (`state`, `store`, `dept`) are added next to the ids. Use the plain names on every chart; keep the ids for joins.

In [ ]:
sales = data.add_readable_names(data.load_sales_foods())
cal = data.load_calendar()
days = data.day_columns(sales)
dates = cal.set_index('d').loc[days, 'date'].values      # d_1 ... d_1941 -> dates
sales.shape

## Carried over: per-series helpers from yesterday
Zero days are counted only from each series' first sale. If these get used again tomorrow, move them into `src/m5food`.

In [ ]:
X = sales[days].to_numpy()
first = (X > 0).argmax(axis=1)                        # first day with a sale
live = np.arange(X.shape[1]) >= first[:, None]        # days from the first sale on
sales['first_sale'] = pd.to_datetime(dates[first])
sales['zero_share'] = ((X == 0) & live).sum(1) / live.sum(1) * 100
sales['mean_daily'] = np.nanmean(np.where(live, X, np.nan), axis=1)
sales[['item_id', 'store', 'dept', 'first_sale', 'zero_share', 'mean_daily']].head()

## Today

### Simple ACF: total FOODS, raw
ACF at lag N = how much today's sales look like sales N days earlier. 1 = same pattern, 0 = no link.

Look for: tall bars at 7, 14, 21 (weekly)? A bump near 30 (monthly)? Something at 365 (yearly)? Hover a bar to read its value; drag over 0-40 to zoom in.

In [ ]:
from statsmodels.tsa.stattools import acf

total = pd.Series(X.sum(axis=0), index=pd.DatetimeIndex(dates), name='All FOODS')   # units per day, all stores
corr = acf(total, nlags=400, fft=True)                                               # corr[N] = link to N days ago
noise = 1.96 / np.sqrt(len(total))                                                   # inside this band = could be chance

fig = go.Figure(go.Bar(x=np.arange(len(corr)), y=corr, marker_color=plots.PALETTE[0],
                       hovertemplate='today vs %{x} days ago: %{y:.2f}<extra></extra>'))
fig.add_hrect(y0=-noise, y1=noise, fillcolor='#ddd', opacity=0.6, line_width=0)
for lag, name in {7: '1 week', 14: '2 weeks', 30: '~1 month', 365: '1 year'}.items():
    fig.add_vline(x=lag, line=dict(color='#999', width=1, dash='dot'),
                  annotation_text=name, annotation_position='top')
fig.update_layout(title='<b>All FOODS: how much does today look like N days ago?</b>',
                  xaxis_title='N = days ago (lag)', yaxis_title='correlation (1 = same pattern, 0 = no link)',
                  yaxis_range=[-0.3, 1.05], template='simple_white', height=500, bargap=0)
fig.show()

My obs:
- I see a strong weekly pattern 7,14,21,28
- lag 1 is also strong which means busy day tends to a busy day , it directly doesnt die out 
- i cant see yearly acf properperly next to divide those

In [ ]:
from plotly.subplots import make_subplots

# Clean-up helpers used by the charts below
def fill_christmas(s):
    """Stores are closed on 25 Dec (about 15 units instead of ~30,000). Use the average of the day before and after."""
    s = s.astype(float).copy()
    for d in s.index[(s.index.month == 12) & (s.index.day == 25)]:
        s[d] = (s[d - pd.Timedelta(days=1)] + s[d + pd.Timedelta(days=1)]) / 2
    return s

def remove_trend(s):
    """Subtract a straight-line trend fitted over the whole period."""
    t = np.arange(len(s))
    slope, intercept = np.polyfit(t, s.to_numpy(dtype=float), 1)
    return s - (slope * t + intercept)

def remove_weekday(s):
    """Subtract the average of each weekday, so the weekly pattern is gone and other cycles can show."""
    return s - s.groupby(s.index.dayofweek).transform('mean')

def acf_bars(values, title, highlight=None, max_lag=None, marks=None, height=420):
    """ACF as bars. highlight(lag) -> True colours that bar; marks = {lag: label} draws dotted guide lines."""
    lags = np.arange(len(values)) if max_lag is None else np.arange(max_lag + 1)
    vals = values[:len(lags)]
    colors = [plots.PALETTE[0] if (highlight and highlight(l)) else '#9aa5ad' for l in lags]
    fig = go.Figure(go.Bar(x=lags, y=vals, marker_color=colors,
                           hovertemplate='today vs %{x} days ago: %{y:.2f}<extra></extra>'))
    fig.add_hrect(y0=-noise, y1=noise, fillcolor='#ddd', opacity=0.6, line_width=0)
    for lag, name in (marks or {}).items():
        fig.add_vline(x=lag, line=dict(color='#777', width=1, dash='dot'),
                      annotation_text=name, annotation_position='top')
    fig.update_layout(title=title, xaxis_title='N = days ago (lag)', yaxis_title='correlation',
                      yaxis_range=[-0.4, 1.05], template='simple_white', height=height, bargap=0.15)
    return fig

clean = remove_trend(fill_christmas(total))
corr_clean = acf(clean, nlags=420, fft=True)

### Zoom: the first 8 weeks, cleaned
Christmas closures filled and the 5-year trend removed. **Blue bars = multiples of 7 days.** Look for: are the blue bars clearly taller than the grey ones?

In [ ]:
fig = acf_bars(corr_clean, '<b>All FOODS, first 8 weeks: blue = same weekday</b>', max_lag=56,
               highlight=lambda l: l > 0 and l % 7 == 0, marks={7: '1 wk', 14: '2 wk', 28: '4 wk'})
fig.show()

weekday_share = remove_trend(fill_christmas(total)).groupby(total.index.dayofweek).transform('mean')
print(f"Weekly pattern explains {weekday_share.var() / clean.var():.0%} of day-to-day variation (after trend removed)")
by_day = fill_christmas(total).groupby(total.index.day_name()).mean()
print((by_day / by_day.mean() * 100).round(0).reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday',
                                                       'Friday', 'Saturday', 'Sunday']).astype(int).to_string())

**Draft observations**
- Every blue bar (same weekday) is tall and every grey bar is low. Lag 7 = 0.72, lag 3 = -0.09. The weekly beat is the strongest pattern in FOODS.
- The weekly pattern alone explains about half (51%) of the day-to-day variation once the trend is removed. Weekend days sell about 120% of an average day, Tuesday to Thursday about 88%.
- The blue bars barely shrink from 1 week to 8 weeks, so the weekly shape is stable, not drifting. A model can treat it as a fixed weekday effect.

### What is left once the weekly pattern is removed?
Same cleaned series, minus the average of each weekday. Dotted lines at 1, 2 and 3 months and at 1 year. Look for: a cycle that was hidden behind the weekly comb.

In [ ]:
corr_noweek = acf(remove_weekday(clean), nlags=400, fft=True)
fig = acf_bars(corr_noweek, '<b>All FOODS with the weekly pattern removed</b>',
               highlight=lambda l: any(abs(l - m) <= 2 for m in (30, 61, 91, 122, 152, 182, 213, 243, 274, 304, 335, 365)),
               marks={30: '1 month', 61: '2 months', 91: '3 months', 365: '1 year'})
fig.show()
print(f"lag 30 before removing the weekly pattern: {corr_clean[30]:.2f}, after: {corr_noweek[30]:.2f}")

**Draft observations**
- Once the weekly correlation is gone, a **monthly wave** appears. Lag 30 jumps from 0.16 to 0.61, and the bars rise and fall about every 30 days (blue = near a whole month).
- A monthly cycle in a supermarket points to something paid monthly. M5 has one in the calendar: SNAP days. The next chart checks it.

### Is the monthly wave SNAP?
SNAP = US food-aid payments. On SNAP days, benefit holders can spend their monthly food money. Each state has its own SNAP days (`snap_CA`, `snap_TX`, `snap_WI` in the calendar).
Look for: do sales rise on SNAP days, and is the rise the same in every state?

In [ ]:
by_state_day = sales.groupby('state')[days].sum().T
by_state_day.index = total.index
cal_by_date = cal.set_index('date').loc[total.index]
STATE_CODE = {'California': 'CA', 'Texas': 'TX', 'Wisconsin': 'WI'}

rows = []
for st, code in STATE_CODE.items():
    s = remove_weekday(remove_trend(fill_christmas(by_state_day[st])))   # same cleaning, weekly pattern removed
    snap = cal_by_date[f'snap_{code}'].to_numpy() == 1
    lift = (s[snap].mean() - s[~snap].mean()) / by_state_day[st].mean() * 100
    rows.append(dict(state=st, lift=lift, snap_days=', '.join(map(str, sorted(set(total.index[snap].day))))))
snap_tbl = pd.DataFrame(rows)

fig = go.Figure(go.Bar(x=snap_tbl.state, y=snap_tbl.lift, marker_color=plots.PALETTE[:3],
                       text=snap_tbl.lift.map(lambda v: f'+{v:.0f}%'), textposition='outside',
                       customdata=snap_tbl.snap_days,
                       hovertemplate='%{x}: SNAP days sell %{y:.1f}% more<br>SNAP days of month: %{customdata}<extra></extra>'))
fig.update_layout(title='<b>How much more FOODS sells on SNAP days than on other days</b>',
                  yaxis_title='% above a normal day (weekly pattern removed)', yaxis_range=[0, snap_tbl.lift.max() * 1.25],
                  template='simple_white', height=420)
fig.show()
snap_tbl

**Draft observations**
- Sales rise on SNAP days in every state, but by very different amounts: Wisconsin about +27%, Texas about +15%, California about +10%.
- California's SNAP days are the 1st to 10th of the month. Texas and Wisconsin spread theirs over the 1st to 15th. Either way they come back every month, which matches the monthly wave in the previous chart.
- So the "monthly" cycle is mostly SNAP. Because each state has its own days and strength, it has to be modelled per state, not as one shared monthly effect.

### Is there a real yearly cycle, or only weekly + monthly repeating?
Only lags that are whole weeks are shown (308 to 420 days), so the weekly comb cannot fool us. **Blue = whole multiples of 4 weeks (28 days)**, roughly a month.
Look for: is lag 364 (1 year) clearly taller than its neighbours?

In [ ]:
week_lags = np.arange(280, 421, 7)
fig = go.Figure(go.Bar(x=week_lags, y=corr_clean[week_lags],
                       marker_color=[plots.PALETTE[0] if l % 28 == 0 else '#9aa5ad' for l in week_lags],
                       text=[f'{v:.2f}' for v in corr_clean[week_lags]], textposition='outside',
                       hovertemplate='today vs %{x} days ago: %{y:.2f}<extra></extra>'))
fig.add_vline(x=364, line=dict(color='#777', dash='dot'), annotation_text='1 year (364)', annotation_position='top')
fig.update_layout(title='<b>Whole-week lags around one year: blue = multiples of 4 weeks</b>',
                  xaxis_title='N = days ago (lag)', yaxis_title='correlation', yaxis_range=[0, 0.8],
                  template='simple_white', height=420)
fig.show()

month_of_year = clean.groupby(clean.index.month).transform('mean')
print(f"Calendar month explains {month_of_year.var() / clean.var():.0%} of day-to-day variation (weekday pattern: about 51%)")

My obs:
- Conclusion for now: no strong yearly cycle at total level. Yearly effects may still exist for specific holidays (Christmas, Thanksgiving, Super Bowl) and should be tested as events, not as a smooth season.

### Does every store have the same cycles?
One row per store. Two numbers: the **weekly** link (lag 7) and the **monthly** link once the weekly pattern is removed (lag 30). Both after Christmas fill and trend removal.
Look for: stores that stand out, especially Wisconsin store 1, Wisconsin store 2 and California store 2 (the level-shift stores).

In [ ]:
by_store_day = sales.groupby('store')[days].sum().T
by_store_day.index = total.index
rows = []
for store in by_store_day:
    raw = by_store_day[store].astype(float)
    c = remove_trend(fill_christmas(raw))
    rows.append({'store': store, 'Weekly (lag 7)': acf(c, nlags=7, fft=True)[7],
                 'Monthly (lag 30, weekly removed)': acf(remove_weekday(c), nlags=30, fft=True)[30],
                 'raw lag 100 (level shift check)': acf(raw, nlags=100, fft=True)[100]})
store_acf = pd.DataFrame(rows).set_index('store')
order = store_acf.index[::-1]                       # California store 1 on top

cols = ['Weekly (lag 7)', 'Monthly (lag 30, weekly removed)']
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=cols, horizontal_spacing=0.04)
STATE_COLOR = {'California': plots.PALETTE[0], 'Texas': plots.PALETTE[1], 'Wisconsin': plots.PALETTE[2]}
for j, col in enumerate(cols, start=1):
    fig.add_trace(go.Bar(y=order, x=store_acf.loc[order, col], orientation='h',
                         marker_color=[STATE_COLOR[s.split(' store ')[0]] for s in order],
                         text=store_acf.loc[order, col].map('{:.2f}'.format), textposition='outside',
                         hovertemplate='%{y}: %{x:.2f}<extra></extra>', showlegend=False), row=1, col=j)
    fig.update_xaxes(range=[-0.2, 1.05], row=1, col=j)
fig.update_layout(title='<b>Cycle strength per store</b> <span style="font-size:13px;color:#444">'
                        '(0 = no link, 1 = identical pattern)</span>',
                  template='simple_white', height=520)
fig.show()
store_acf.round(2)

**Draft observations**
- **Weekly:** every store has it (0.36 to 0.86). It is weakest in Wisconsin store 2 (0.36) and Texas store 3 (0.54), and strongest in California store 2 (0.86).
- **Monthly (SNAP):** once the weekly pattern is removed, every store has a monthly wave (0.26 to 0.69). It is strongest in Wisconsin store 2 (0.69) and Wisconsin store 3 (0.66). It is weakest in Wisconsin store 1 (0.26), which is also a level-shift store, so that low value may be distorted.
- **Level shifts:** Wisconsin store 1 and Wisconsin store 2 have high raw lag-100 values (0.48 and 0.38) because their sales jumped to a new level in 2012. Removing a straight-line trend does not fully fix a jump. That is probably why Wisconsin store 2's weekly number is so low. These stores need the jump handled before their ACF can be trusted.

### ACF per state: weekly and monthly
Same clean-up as above (Christmas filled, trend removed). **Left:** first 8 weeks, coloured bars = same weekday. **Right:** weekly pattern removed, first ~3 months, to see the monthly (SNAP) wave.
Look for: does each state peak near 30 days, and how deep is the dip in between?

In [ ]:
# ACF per state: left = weekly (cleaned, first 8 weeks), right = monthly (weekly pattern removed, first ~3 months)
by_state_day = sales.groupby('state')[days].sum().T
by_state_day.index = total.index
STATES = ['California', 'Texas', 'Wisconsin']
WEEK_MAX, MONTH_MAX = 56, 100
fig = make_subplots(rows=3, cols=2, shared_yaxes=True, vertical_spacing=0.09, horizontal_spacing=0.05,
                    subplot_titles=[f'{st}: {t}' for st in STATES
                                    for t in ('weekly (coloured = same weekday)', 'monthly (weekly pattern removed)')])
state_acf = {}
for i, st in enumerate(STATES, start=1):
    c = remove_trend(fill_christmas(by_state_day[st]))
    wk = acf(c, nlags=WEEK_MAX, fft=True)
    mo = acf(remove_weekday(c), nlags=MONTH_MAX, fft=True)
    state_acf[st] = {'weekly (lag 7)': wk[7], 'monthly (lag 30)': mo[30], 'lag 15 (half month)': mo[15],
                     'lag 61 (2 months)': mo[61], 'lag 91 (3 months)': mo[91]}
    lags = np.arange(WEEK_MAX + 1)
    fig.add_trace(go.Bar(x=lags, y=wk, marker_color=[STATE_COLOR[st] if l % 7 == 0 and l else '#c4ccd2' for l in lags],
                         hovertemplate=st + '<br>today vs %{x} days ago: %{y:.2f}<extra></extra>'), row=i, col=1)
    lags = np.arange(MONTH_MAX + 1)
    fig.add_trace(go.Bar(x=lags, y=mo, marker_color=STATE_COLOR[st],
                         hovertemplate=st + '<br>today vs %{x} days ago: %{y:.2f}<extra></extra>'), row=i, col=2)
    for col in (1, 2):
        fig.add_hrect(y0=-noise, y1=noise, fillcolor='#ddd', opacity=0.6, line_width=0, row=i, col=col)
    for lag in (30, 61, 91):
        fig.add_vline(x=lag, line=dict(color='#777', width=1, dash='dot'), row=i, col=2)
fig.update_yaxes(range=[-0.4, 1.05])
fig.update_xaxes(title_text='N = days ago (lag)', row=3)
fig.update_layout(title='<b>ACF per state: weekly and monthly cycles</b><br>'
                        '<span style="font-size:13px;color:#444">Christmas filled, trend removed. Dotted lines on '
                        'the right: 30, 61 and 91 days (1, 2, 3 months). Grey band: could be pure chance.</span>',
                  template='simple_white', height=900, showlegend=False, bargap=0.15)
fig.show()
pd.DataFrame(state_acf).T.round(2)

**Draft observations (check against the chart)**
- **Weekly:** all three states have the 7-day comb. It is strongest in California (0.79), then Texas (0.65), weakest in Wisconsin (0.49). Wisconsin is probably pulled down by Wisconsin store 2's level jump.
- **Monthly:** all three peak at about one month (California 29 days, Texas 30, Wisconsin 31). But the shape differs. **Wisconsin has a real wave**: it drops below zero at half a month (−0.16) and comes back at 1, 2 and 3 months (0.59, 0.58, 0.49). **Texas dips less** (0.11 at half a month). **California barely dips** (0.38), so its monthly link is weaker and partly just slow drift.
- This matches the SNAP lift: Wisconsin +27%, Texas +14%, California +10%. **The stronger the SNAP lift, the clearer the monthly wave.** That points to SNAP being modelled per state.

### Does the weekly cycle survive down to single items?
We forecast each item in each store, not totals. One lag-7 value per item-store series (from its first sale), grouped by how fast the item sells.
Look for: does the weekly link fade as items sell less?

### ACF summary: inputs for my EDA doc
**These are drafts built from the numbers above. Check each against the charts, then rewrite them in your own words.**

**Entry 11. ACF at total, state and store level**
- **Entry title (draft):** FOODS sales repeat every week everywhere, every month mainly through SNAP, and barely every year.
- **Level:** Total → State → Store → Item
- **What I saw:** a strong 7-day comb in the ACF. A ~30-day wave once the weekly pattern is removed. No yearly peak above the 4-week neighbours.
- **Evidence (numbers):** lag 7 = 0.72 (cleaned total). Weekday pattern explains about 51% of variation, calendar month about 2%. Lag 30 rises from 0.16 to 0.61 once the weekly pattern is removed. SNAP-day lift: Wisconsin +27%, Texas +15%, California +10%. Lag 364 = 0.57 vs 336 = 0.55.
- **Status:** Checked (numbers computed above).
- **Drilled down into:** stores (weekly 0.36 to 0.86, monthly 0.26 to 0.69 once the weekly pattern is removed), items (median weekly link 0.09 for under 0.5/day up to 0.47 for 10+/day).
- **Came back up:** the weekly beat in totals is real, but at item level it is weak for slow sellers. Any yearly "season" seen in totals is mostly weekly + monthly rhythm.
- **Next question:** for slow items, does borrowing the weekly shape from the store or department beat learning it per item?

**Also feeds:** Entry 5 (weekly pattern: weekend about 120%, Tue-Thu about 88%), Entry 7 (SNAP: the monthly wave, strongest in Wisconsin), Entry 4 (Wisconsin store 1 and 2 level shifts distort their ACF).

**Decision candidates (for the Decisions log; yours to agree or reject)**
| Decision | Why this | Why not the others | Status |
|---|---|---|---|
| Model the weekday as a fixed effect | 51% of variation, stable over 8 weeks | A smooth curve would miss the sharp weekend jump | My guess |
| Model SNAP per state, not as one monthly effect | Lift ranges from +10% (California) to +27% (Wisconsin), and the SNAP days differ | One shared effect would over-predict California and under-predict Wisconsin | My guess |
| No smooth yearly season for now; test holidays as events | Lag 364 is no higher than nearby 4-week lags; month explains about 2% | A yearly season would add parameters for little gain | My guess |
| Handle the Wisconsin store 1/2 level shift before modelling | Raw lag-100 of 0.48/0.38 shows the jump distorts patterns | Ignoring it biases the level and the ACF for those stores | My guess |

**Questions for Chiranjit (drafts)**
- Are SNAP days still a relevant driver for the business case, or is that M5-specific?
- For slow items, is it acceptable to share the weekly shape from store or department level?

**Terms I had to learn**
| Term | Plain meaning |
|---|---|
| ACF (autocorrelation) | How much today's sales look like sales N days earlier |
| Lag | The N in "N days earlier" |
| Noise band | The grey zone where a value could be pure chance |
| Detrend | Remove the slow long-term rise so it does not lift every lag |
| SNAP | US food-aid payments. On SNAP days people can spend their monthly food money |
| Level shift | A sudden, lasting jump in a store's sales level |

### Is each store stationary? Four checks
**Stationary** = the average and the spread stay the same over time, so the past is a fair guide to the future.
No single number proves it. Run four checks on total FOODS plus the 10 stores and see whether they agree:

| # | Check | Stationary looks like |
|---|---|---|
| 1 | 90-day rolling average and spread | both lines flat |
| 2 | First half vs second half | nearly equal |
| 3 | ACF fade | drops near zero within a few weeks |

Christmas closures are filled first (a closed day is not a real drop).

In [ ]:
# Stationarity set-up: total FOODS + the 10 stores, Christmas closures filled (a closed day is not a real drop)
import warnings
from statsmodels.tsa.stattools import adfuller, kpss

series = {'All FOODS': fill_christmas(total)}
for store in sorted(by_store_day.columns):
    series[store] = fill_christmas(by_store_day[store])
NAMES = list(series)                     # All FOODS first, then California, Texas, Wisconsin stores
# Own colour map, so re-running other cells (which redefine STATE_COLOR) cannot break it
SERIES_COLOR = {'All': '#333333', 'California': plots.PALETTE[0], 'Texas': plots.PALETTE[1], 'Wisconsin': plots.PALETTE[2]}
def color_of(name):
    return SERIES_COLOR[name.split(' ')[0]]       # 'Texas store 2' -> 'Texas', 'All FOODS' -> 'All'
print(len(NAMES), 'series,', len(total), 'days each')

In [ ]:
# Check 1: 90-day rolling average and 90-day spread. Index 100 = the series' own 5-year average, so all panels share one scale.
fig = make_subplots(rows=4, cols=3, shared_xaxes=True, shared_yaxes=True, subplot_titles=NAMES,
                    vertical_spacing=0.06, horizontal_spacing=0.04)
for i, name in enumerate(NAMES):
    s = series[name]; base = s.mean()
    r, c = i // 3 + 1, i % 3 + 1
    fig.add_trace(go.Scatter(x=s.index, y=s.rolling(90).mean() / base * 100, mode='lines',
                             line=dict(color=color_of(name), width=2), name='90-day average',
                             legendgroup='avg', showlegend=(i == 0),
                             hovertemplate=name + '<br>%{x|%d %b %Y}<br>average: %{y:.0f}<extra></extra>'), row=r, col=c)
    fig.add_trace(go.Scatter(x=s.index, y=s.rolling(90).std() / base * 100, mode='lines',
                             line=dict(color=color_of(name), width=1.5, dash='dot'), name='90-day spread (std)',
                             legendgroup='std', showlegend=(i == 0),
                             hovertemplate=name + '<br>%{x|%d %b %Y}<br>spread: %{y:.0f}<extra></extra>'), row=r, col=c)
    fig.add_hline(y=100, line=dict(color='#bbb', width=1), row=r, col=c)
fig.update_layout(title='<b>Check 1: do the 90-day average and spread stay flat?</b><br>'
                        '<span style="font-size:13px;color:#444">Solid = average, dotted = spread. '
                        '100 = that series\' own 5-year average. Flat lines = stationary.</span>',
                  template='simple_white', height=950, legend=dict(orientation='h', y=-0.05))
fig.show()

**Draft observations (check against the chart)**
- Most stores' 90-day average wanders between about **70 and 120** of their own 5-year average. It climbs slowly and dips each winter. Not flat, but not wild.
- **Wisconsin stores 1 and 2 roughly triple**: from about 45 to about 150. That is a level jump, not a slow drift.
- The spread (dotted) mostly stays between 10 and 25. **California store 2** is the exception: its spread rises to 40 in the later years, so its sales got jumpier.

In [ ]:
# Check 2: split in half. Compare the first 970 days with the last 971 days.
half = len(total) // 2
rows = []
for name in NAMES:
    a, b = series[name].iloc[:half], series[name].iloc[half:]
    rows.append({'series': name,
                 'avg 1st half': a.mean(), 'avg 2nd half': b.mean(),
                 'avg change %': (b.mean() / a.mean() - 1) * 100,
                 'spread 1st half': a.std(), 'spread 2nd half': b.std(),
                 'spread change %': (b.std() / a.std() - 1) * 100})
halves = pd.DataFrame(rows).set_index('series')

fig = go.Figure()
for col, dash, sym in [('avg change %', None, 'circle'), ('spread change %', None, 'diamond')]:
    fig.add_trace(go.Scatter(y=halves.index[::-1], x=halves[col][::-1], mode='markers+text', name=col.replace(' %', ''),
                             marker=dict(size=12, symbol=sym), text=halves[col][::-1].map('{:+.0f}%'.format),
                             textposition='middle right',
                             hovertemplate='%{y}<br>' + col + ': %{x:+.1f}%<extra></extra>'))
fig.add_vrect(x0=-10, x1=10, fillcolor='#ddd', opacity=0.5, line_width=0,
              annotation_text='within ±10% = nearly equal', annotation_position='top left')
fig.update_layout(title='<b>Check 2: second half vs first half</b>',
                  xaxis_title='change from first half to second half (%)', template='simple_white',
                  height=520, legend=dict(orientation='h', y=-0.15))
fig.show()
halves.round(1)

**Draft observations**
- All FOODS sells **13% more** per day in the second half than in the first. Most stores grow 3–17%.
- **Wisconsin store 1 (+76%) and Wisconsin store 2 (+55%)** are far from equal. Two stores fell: **Wisconsin store 3 (−14%)** and Texas store 2 (−6%).
- Spread is mostly a bit lower in the second half (−3% to −24%). **California store 2 is the outlier: +78% spread** while its average rose only 17%.

In [ ]:
# Check 3: how fast does the raw link to the past fade? Only same-weekday lags (7, 14, 21 ...), so the weekly dip does not fool us.
# Stationary = falls to near zero within a few weeks.
WEEK_LAGS = np.arange(7, 400, 7)
rows, fig = [], go.Figure()
for name in NAMES:
    a = acf(series[name], nlags=400, fft=True)
    below = WEEK_LAGS[a[WEEK_LAGS] < 0.2]
    rows.append({'series': name, 'lag 7': a[7], 'lag 28': a[28], 'lag 98': a[98], 'lag 364': a[364],
                 'first weekly lag below 0.2': int(below[0]) if len(below) else '>400'})
    fig.add_trace(go.Scatter(x=WEEK_LAGS, y=a[WEEK_LAGS], mode='lines', name=name,
                             line=dict(color=color_of(name), width=3.5 if name == 'All FOODS' else 1.4),
                             hovertemplate=name + '<br>same weekday, %{x} days ago: %{y:.2f}<extra></extra>'))
acf_fade = pd.DataFrame(rows).set_index('series')
fig.add_hrect(y0=-noise, y1=noise, fillcolor='#ddd', opacity=0.6, line_width=0)
fig.add_hline(y=0.2, line=dict(color='#999', dash='dot'), annotation_text='0.2 = weak link', annotation_position='right')
fig.update_layout(title='<b>Check 3: does the link to the same weekday in the past fade?</b><br>'
                        '<span style="font-size:13px;color:#444">Raw series. Colour = state (blue California, '
                        'orange Texas, green Wisconsin, thick black = all FOODS). Click a legend name to hide it.</span>',
                  xaxis_title='N = days ago (only whole weeks)', yaxis_title='correlation', yaxis_range=[-0.3, 1.05],
                  template='simple_white', height=520)
fig.show()
acf_fade.round(2)

**Draft observations**
- Even **14 weeks back (lag 98)** the link to the same weekday is still 0.40–0.75. A stationary series would fall near zero within a few weeks.
- **7 of 11 series never drop below 0.2** in 57 weeks. The slowest fade is in Wisconsin store 1 (0.75 at lag 98), which matches its level jump.
- A slow fade like this means *the level moves*: sales today look like sales months ago mainly because both sit on the same slowly rising level.

### Stationarity summary: inputs for my EDA doc
**Drafts from the numbers above. Check them and rewrite in my own words.**

**Finding:** Raw daily sales are **not stationary** in any store: the level rises over 5 years (total +13% second half vs first) and the link to past weeks fades very slowly. All four checks agree. **Comparing each day with the same weekday last week makes every store stationary.**

**Evidence:** 90-day average ranges 70–120 (Wisconsin stores 1 and 2: 45 to 150); halves differ up to +76%; same-weekday ACF still 0.40–0.75 at lag 98; ADF/KPSS non-stationary raw for 10 of 11, stationary weekly-differenced for 11 of 11.

**Decision candidates** *(My guess, not decided)*

| Option | Why this | Why not |
|---|---|---|
| Use weekly differencing (lag 7) for classical models like ARIMA | Fixes the level and the weekly cycle in one step; all 11 pass | Hides the Wisconsin jumps; forecasts must be added back up |
| Use log only | Calms big swings | Does not fix the level problem; tests still fail |
| Tree models with lag and rolling-average features | They don't need stationarity if the features carry the level | Can't predict beyond levels seen in training |
| Wisconsin stores 1 and 2: train only on data after the jump, or add a "level period" flag | The old level is no longer true | Fewer days left to learn yearly patterns |

**Question for Chiranjit (draft):** Do we need stationarity at all? That depends on whether the baseline is ARIMA-style or tree-based. Also: is anything known about what happened to Wisconsin stores 1 and 2 (remodel, expansion)?

**Terms:** *stationary* = average and spread stay the same over time · *ADF* = test that starts by assuming "not stationary" · *KPSS* = test that starts by assuming "stationary" · *weekly differencing* = today minus the same weekday last week · *level shift* = a lasting jump in normal sales.

### Check yourself
ADF p = 0.40 and KPSS p = 0.01. What is the verdict, and what do you do next?

*My answer:* 

## What to decide here
*(fill in at the end of the day)*